# 058 — Types of RNN Architecture

One-to-many, many-to-one, many-to-many: these sound like different
architectures. They are not. All of them run **the same cell with the same
weights** — what differs is which hidden states you keep and what you feed back
in. Demonstrated below with **400 parameters** throughout.

| Part | What we check |
|---|---|
| A | one cell, 400 parameters |
| B | four wirings, four input/output shapes, the same weights |
| C | encoder-decoder, where the lengths need not match |
| D | one Keras argument switches the type |

## Part A — The cell

In [ ]:
import numpy as np

F, H, T = 8, 16, 5
rng = np.random.default_rng(0)
Wx = rng.normal(0, 0.3, (F, H))
Wh = rng.normal(0, 0.3, (H, H))
b = np.zeros(H)
PARAMS = Wx.size + Wh.size + b.size

def cell(x, h):
    return np.tanh(x @ Wx + h @ Wh + b)

print(f"one SimpleRNN cell: F={F}, H={H} -> {PARAMS} parameters")
print(f"  {F}x{H} input + {H}x{H} recurrent + {H} bias")
assert PARAMS == F * H + H * H + H == 400
X = rng.normal(0, 1, (T, F))
print(f"input sequence: {X.shape}")

## Part B — Four wirings of that one cell

In [ ]:
# many-to-one: keep only the last state
h = np.zeros(H)
for t in range(T):
    h = cell(X[t], h)
many_to_one = h

# many-to-many: keep every state
h, outputs = np.zeros(H), []
for t in range(T):
    h = cell(X[t], h)
    outputs.append(h)

# one-to-many: feed the output back in as the next input
h, seed, generated = np.zeros(H), X[0], []
for t in range(T):
    h = cell(seed, h)
    generated.append(h)
    seed = h @ np.linalg.pinv(Wx)

print(f"  many-to-one   {T} in, 1 out    keep only the last state")
print(f"  many-to-many  {T} in, {len(outputs)} out    keep every state")
print(f"  one-to-many   1 in, {len(generated)} out    feed the output back")
assert many_to_one.shape == (H,)
assert len(outputs) == T and len(generated) == T
assert np.array_equal(outputs[-1], many_to_one)      # the last state is the same state
print()
print("many-to-one and many-to-many ran the SAME loop - one keeps the list.")

## Part C — Encoder-decoder, where the lengths differ

In [ ]:
h = np.zeros(H)
for t in range(T):
    h = cell(X[t], h)                 # the encoder emits nothing at all
context = h

decoded, state = [], context
for t in range(3):                    # three outputs from five inputs
    state = cell(np.zeros(F), state)
    decoded.append(state)

print(f"  encoder: {T} steps in, 0 out   -> one context vector {context.shape}")
print(f"  decoder: {len(decoded)} steps out")
print(f"  {T} in, {len(decoded)} out - the lengths need not match")
assert len(decoded) == 3 and context.shape == (H,)
print()
print(f"All four wirings used the same {PARAMS} parameters:")
print(f"  Wx {Wx.shape} + Wh {Wh.shape} + b {b.shape} = {PARAMS}")
assert Wx.size + Wh.size + b.size == 400
print("The type is the wiring, not the cell.")

## Part D — Which is why Keras needs one argument

In [ ]:
BATCH = 32
print(f"  return_sequences=False -> (batch, {H})       last state   -> many-to-one")
print(f"  return_sequences=True  -> (batch, {T}, {H})    every state -> many-to-many")
print(f"  parameter count is {PARAMS} either way")

shapes = {False: (BATCH, H), True: (BATCH, T, H)}
for flag, shape in shapes.items():
    print(f"  return_sequences={str(flag):<5} output shape {shape}")
assert shapes[False] != shapes[True]
assert shapes[True][1] == T

print()
print("Match the type to your data: is the input a sequence, and is the output")
print("one? Sentiment is many-to-one. Part-of-speech tagging is many-to-many of")
print("equal length. Translation is many-to-many of unequal length, which needs")
print("the encoder-decoder of Part C - and lesson 082 builds exactly that.")

## What this notebook established

- One `SimpleRNN` cell with $F=8$, $H=16$ has **400** parameters: $8{\times}16$
  input, $16{\times}16$ recurrent, 16 bias.
- Four "types" ran the same cell with the same 400 parameters. Many-to-one and
  many-to-many are literally the same loop; one keeps the list of states.
- An encoder-decoder emits nothing while encoding, which is what lets the input
  and output lengths differ.
- `return_sequences` switches the output shape between `(batch, 16)` and
  `(batch, 5, 16)` and changes no parameters.

## Exercises

1. Build a many-to-many of **unequal** length without an encoder-decoder. What
   goes wrong?
2. In the one-to-many loop, `pinv(Wx)` fakes the projection back to input space.
   What would a real text generator use instead?
3. Stack two recurrent layers (the first with `return_sequences=True`). Write
   the parameter count for both.